# TM1py: A Mental Model

This module is the first encounter with tm1py, the Python library for TM1
and Planning Analytics. Readers are assumed to be expert in TM1, fluent
with cubes, dimensions, hierarchies, MDX, TurboIntegrator, and chores, and
to have only passing acquaintance with Python. The aim is not to write
code yet. The aim is to install three ideas about what kind of thing
tm1py is, so that everything that follows can be predicted from those
ideas instead of memorized one method at a time.

The chunks that come after this one introduce the API in the order most
TM1 administrators meet it: connecting to a server, navigating the model,
reading cells, writing cells, building objects, deploying processes. Each
of those chunks assumes the mental model from this one is in place. A
learner who skips this chunk and starts with code will eventually arrive
at the same understanding by accident, after some number of confusing
bugs. The point of doing it first is to skip the bugs.

There is deliberately no code in this module. The three ideas concern
what kind of thing tm1py is, not which method to call. Code arrives in
chunk 2.

The topics below are arranged linearly for review. Structural grouping
(sections, chapters) can be applied later.

---

## Topic list

1. tm1py is a thin wrapper over the TM1 REST API
2. The library has two kinds of class: Objects and Services
3. The server is the source of truth; Python objects are snapshots
4. What this lets you predict
5. Common misunderstandings

---

## 1. tm1py is a thin wrapper over the TM1 REST API

The first idea is what tm1py is, in one phrase: a thin wrapper over the
TM1 REST API. Every word of that phrase carries weight, and the rest of
the topic unpacks each one.

Every TM1 server since version 11 exposes a REST API, served from the
same process that hosts the cubes and rules. That API is the canonical
programmatic interface to TM1. Anything that PAW does, anything that
Architect does, anything an external client does, comes down to a sequence
of HTTP requests against that endpoint. The tm1py library is one such
external client, written in Python.

"Thin" means the library does not add behaviour. It marshals Python
arguments into an OData URL and a JSON body, sends the request, parses
the response, and returns the result as Python data. There is no
in process simulation of cube behaviour, no compiled query engine, no
local emulation of rules or feeders. If the TM1 server is unreachable,
tm1py is inert. If the REST API does not expose an operation, tm1py
cannot do it either.

Three properties of the library follow directly from this thinness, and
the chunks file calls them out by name.

There is **no caching**. Calling the same read twice produces two HTTP
requests. The library does not remember the first answer to short circuit
the second. Whether the answer would change in the meantime or not is
not the library's concern; the request is sent each time. Caching, if
desired, is the user's responsibility, accomplished by storing the
result in an ordinary Python variable.

There is **no ORM**. The Python objects returned by a read are not
connected to the server after the moment they were returned. Mutating
an object in Python does not propagate any change. There is no
transparent dirty tracking, no automatic flush, no session-managed
identity map. The library returns plain Python data; persisting any
change is a separate, explicit call.

There is **no magic**. Python's dynamic features are used sparingly.
The library has a flat, predictable shape: a connection object, a small
number of services hanging off it, and a small set of verbs on each
service. The shape mirrors the REST endpoints almost one to one. What
the user calls is what the user gets.

The mental anchor for this topic is to imagine the TM1 REST
documentation and the tm1py source code side by side. Every method on
every service has a matching REST endpoint behind it. The library's
job is to make that endpoint pleasant to call from Python. It is not the
library's job to extend or reinterpret what the endpoint can do. This
constraint is a feature: the API surface is exactly as large as TM1's
REST surface, and a question about either can usually be answered by
consulting the other.

## 2. The library has two kinds of class: Objects and Services

The second idea concerns the shape of the codebase. On first reading,
tm1py looks larger than it is, because every TM1 concept appears to have
two classes attached: a `Cube` and a `CubeService`, a `Dimension` and a
`DimensionService`, a `Process` and a `ProcessService`. The doubling
looks redundant until the distinction between the two kinds is made
explicit. Once made, almost every other class in the library sorts
cleanly into one camp or the other.

**Objects are passive.** A `Cube` instance carries the data that
describes a cube: its name, the list of dimensions it spans, its rules
text, its attributes. It serializes cleanly to JSON, because it is in
effect the JSON document the server returned, decoded into a Python
class with named attributes. An Object is built either by the library
from a server response, or by the user, in code, before sending it back.
An Object on its own does not talk to TM1. It does not have a `save`
method, because saving is not the Object's job.

**Services are active.** A `CubeService` instance carries the
connection: the session token, the credentials, the server address, the
HTTP machinery. It exposes verbs that act on cubes by name or by being
handed an Object: `get`, `get_all`, `create`, `update`, `delete`,
`exists`, and a handful of operation-specific extras. Every method on a
service results in an HTTP request. A Service does not carry data about
any specific cube; it carries the means to fetch and modify cubes in
general.

The two interact in a predictable rhythm.

For a read: call a verb on the Service, receive an Object. The Service
has done the work of issuing the request and parsing the response; the
returned Object is the parsed payload, ready for inspection in Python.

For a write: construct or modify an Object, then hand it back to the
Service for `create` or `update`. The Object is the payload to send.
The Service is what actually sends it.

This separation is also why the namespace under `TM1Service` looks the
way it does. The connection facade has a `cubes` attribute, a
`dimensions` attribute, a `processes` attribute, and so on. Each of
those attributes is itself a Service for one TM1 concept. The
`TM1Service` is the connection. The per-concept services are the verbs
against that connection.

The pairing is worth grasping early because it dictates where intuition
transfers and where it does not. Comparing two `Cube` instances, or
serializing one to JSON, or printing one for debugging, is meaningful;
they are data. Comparing two `CubeService` instances, or serializing one,
is not meaningful; they are remote handles. Asking "what does this
Object know?" is the right question; asking "what does this Service
know?" is not, because a Service does not know anything about specific
cubes, only how to fetch them.

Most TM1 concepts have the pair. A small handful do not, by accident of
design or because the concept is itself purely procedural (running a
process, listing live threads, monitoring sessions). When in doubt, the
question to ask is whether the thing in question is data or an action.
Data lives in Objects. Actions live in Services.

## 3. The server is the source of truth; Python objects are snapshots

The third idea is the load bearing one. The previous topic established
that an Object is just data, decoupled from the server after the moment
it was fetched. This topic is the practical consequence of that
decoupling, and it is where most of the bugs people write against tm1py
come from.

A snapshot is what a photograph is to a landscape. The photograph
captures the state of the subject at the instant the shutter opened.
Whatever happens to the subject afterwards is invisible to the
photograph. The photograph remains correct as a record of how things
were and incorrect as a description of how things are. The two are
different things, and they diverge as soon as the subject changes.

Every Object that tm1py returns is a photograph in this sense. A `Cube`
returned from a `get` call is the cube as it existed on the server when
the call returned. A `Dimension` is the dimension at that instant. A
list of element names, a subset definition, a view's cellset, a process
script: all of them are snapshots, captured at fetch time, frozen in
memory thereafter. The server's copy continues to evolve, drift,
gain new dimensions, lose subsets. The Python copy does not.

The bug pattern this enables, called out explicitly in the chunks plan
because of how often it occurs, is read–modify–write without locking.
The shape is recognizable: fetch a cube into a Python object, change
something on the local object, call the service's `update` to send it
back. If nobody else touched the cube in between, the pattern works
exactly as expected. If anybody else did touch it, the local object is
out of date when it is sent back, and the `update` call overwrites the
intervening change with the older snapshot. The change is lost
silently. There is no error, no warning. The HTTP request succeeds
because the server has no way to detect that the payload was based on
stale data; from the server's perspective, the request carries
authoritative new state, and authoritative new state is what gets stored.

The mental discipline that prevents this is to think of every Object as
a copy made at a specific instant, valid only at that instant. The
questions that follow are concrete:

How long is the gap between fetching the Object and writing it back? A
millisecond, in a tight script that does no other work? A minute,
because there is some computation in between? An hour, because the
script is interactive and a user is in the loop? The longer the gap,
the more likely the snapshot is stale by the time it returns.

During that gap, who else might be writing to the same object? On a
production cube during business hours, the answer is "users in PAW,
scheduled chores, other scripts." On a development cube at 2am with
nobody else logged in, the answer is "nobody." The risk is a function
of who else has the keys.

What does the right concurrency model look like for this particular
workflow? Read–modify–write of a whole Object is the bluntest tool. A
narrower cell level update, a TI process running on the server (which
is naturally serialized with other server-side work), or a sandbox-based
staging that is committed atomically are all alternatives. None is
universally correct; each suits a different situation.

The same principle applies beyond the explicit `Cube`, `Dimension`,
`Process` Objects. A dimension's element list, fetched into a Python
set, is a snapshot. A view's cellset, pulled into a DataFrame, is a
snapshot. A subset definition, captured for inspection, is a snapshot.
None of them update themselves. The library does not lie about this. It
never claimed to be live. But the natural intuition that comes from
working with familiar Python objects is that they behave like local data
structures, which they do, with one caveat: their relationship to the
server is captured at one moment and frozen thereafter.

The phrase to repeat to oneself is short. Snapshot, not live reference.
Internalizing this early is what separates code that works on quiet
weekend runs from code that works during a busy quarter close.

## 4. What this lets you predict

The three ideas above are short. The point of this chunk is not their
length but their predictive power. A learner who has internalized them
can answer many questions about tm1py without opening the documentation,
because the answers follow from the model.

Why is method X missing from the library? Because the underlying REST
API does not expose it. tm1py reflects what the server offers, no more.
The library is not a place where missing functionality can be added by
clever Python code; if the operation is not available over REST, it is
not available in tm1py.

Why is operation Y slow? Because it makes more HTTP calls than it needs
to. Look at the loop, count the round trips, and the answer is usually
in front of you. The fix is almost always to batch: one large request
that returns many cells, instead of many small requests that each return
one. Performance in tm1py is performance in HTTP.

Why did the change disappear? Because somebody else, or another script,
or a chore, wrote the same Object between the time it was read into
Python and the time it was sent back. The Python snapshot was stale.
The server accepted the stale payload because it cannot tell stale from
fresh; only the caller can.

Why does the read method return an Object that has to be passed back
explicitly to write? Because Objects are passive data. The Object exists
for inspection and modification in Python. The Service is what actually
persists changes. The two are deliberately separate, so that a read of
an Object never accidentally writes anything, and a write to the server
is always an explicit handoff.

Why are there separate `cubes`, `dimensions`, `processes` services on
the connection? Because each one is a thin wrapper around a different
REST endpoint. The split is not arbitrary; it is the shape of the
underlying API. Once that shape is internalized, finding any operation
becomes a matter of knowing which TM1 concept it concerns, then looking
on the Service for that concept.

Why does tm1py not raise when an Object is out of date? Because the
server has no way to know it is out of date. From the server's
perspective, an `update` carries authoritative new data. Detecting that
the payload was constructed from a stale read would require some
optimistic concurrency token (an `ETag`, a version number, a timestamp)
on every Object, and the underlying REST contract does not surface one
in a form that the library could enforce automatically. Detecting
staleness is, therefore, the caller's responsibility.

These predictions are the entire payoff of the mental model. Without it,
the library looks like an arbitrary collection of methods with surprising
performance characteristics and unexplained intermittent bugs. With it,
every behaviour follows from "thin wrapper, two kinds of class, snapshot
semantics," and the documentation reads as confirmation rather than as
discovery.

## 5. Common misunderstandings

A short collection of intuitions that the three ideas above directly
correct. Each is paired with the question to ask in its place.

**"tm1py knows about my cube."** It does not. A `Cube` Object knows
what it was told at the moment of fetch. The Service knows how to
fetch. Nothing in the library tracks server state in the background.
The right question is "when was this Object fetched, and what could
have changed since?"

**"Calling the same getter twice is free."** It is two HTTP round
trips. If the answer would not change for the purpose at hand, fetch
once and store the result in a Python variable. The right question is
"did I already cache this, or am I asking the server again?"

**"If I change the Python Object, the server changes too."** It does
not. Modifying a `Dimension` instance updates a Python data structure;
the server is unaffected until the matching Service's `update` is
called. The right question is "have I sent the change back yet?"

**"If `update` succeeds, my change is correctly applied."** It is
applied as written. If the Object on which the change was based was a
stale snapshot, the change overwrites whatever happened in between. The
right question is "was this Object based on the current state of the
server, or could it be out of date?"

**"tm1py does X for me."** It does whatever the REST API does, packaged
for Python. There is no logic in tm1py that the REST API does not also
support. The right question is "does the REST API support this
operation directly, or would the right approach be a TI process running
on the server?"

These misunderstandings recur because they are the natural intuitions to
bring from spreadsheet work, from object oriented programming generally,
and from libraries that do offer caching, ORMs, or live references. The
cure is the mental model in the first three topics. Every later chunk
in this course assumes that model is in place; with it, the rest of the
library reads as a confirmation of expectations rather than a series of
surprises.